# k04 — Provenance: where did that answer come from?

For one answer, take apart everything the model saw and connect each
piece to its source: the ingested document, the memory store, and the
instruction layer. Nothing here is inferred — it is read back from the
captured model request and the committed trace.

Trust: T2 callback. Network: none (scripted models).

In [ ]:
import tempfile
from pathlib import Path

import finstack_ai
from _knowledge import build_knowledge_agent, scripted_model

workdir = Path(tempfile.mkdtemp(prefix="finstack-know-k04-"))

report = workdir / "holdings.csv"
report.write_text("fund,position,weight\nAlpha Fund,US Treasuries,0.42\n")

## Seed both sources

One run ingests the document (attachment → converted Markdown) and
remembers a fact derived from it. After this cell the data directory
holds two kinds of knowledge: the journaled document turn and the memory
record — each carrying its own provenance.

In [ ]:
seed_script = [
    {
        "text": "",
        "tool_calls": [
            {
                "name": "remember",
                "arguments": {
                    "id": "alpha-treasuries",
                    "keywords": ["alpha", "treasuries", "weight"],
                    "body": "Alpha Fund holds US Treasuries at 42 percent weight (source: holdings.csv).",
                },
            }
        ],
    },
    "Noted: Alpha Fund's largest position is US Treasuries at 42 percent.",
]

seeder = await build_knowledge_agent(workdir, scripted_model(seed_script))
seeded = await seeder.run(
    "A document is attached. Remember the key holding.",
    attachments=[finstack_ai.Attachment(media_type="text/csv", path=str(report))],
)
print(seeded.text)

## One answer, fully attributed

Ask a question, capture the exact model-visible request, and classify
every text block by provenance:

- `[memory]`-suffixed blocks come from the recall provider — records the
  store attributed to earlier runs.
- The system message is the composed instruction layer (base instruction
  plus policy line).
- The user message carries the question.

The scripted answer cites `holdings.csv` — the same name the memory body
carries, so the citation is traceable end to end.

In [ ]:
seen: list[dict] = []


async def _capture(context, request):
    del context
    seen.append(request)
    return {
        "text": "Alpha Fund's largest position is US Treasuries at 42 percent (source: holdings.csv).",
        "completion_id": "k04-answer",
    }


asker = await build_knowledge_agent(
    workdir,
    finstack_ai.PythonModel(
        _capture,
        component="knowledge.model.k04-asker",
        provider="knowledge-scripted",
        model="knowledge-scripted-model",
        context_window_tokens=131_072,
    ),
)
answer = await asker.run("What is Alpha Fund's largest position?")
print(answer.text)

by_provenance: dict[str, list[str]] = {"memory": [], "instructions": [], "user": []}
for message in seen[0]["messages"]:
    for block in message.get("content", []):
        text = block.get("text", "")
        if not text:
            continue
        if text.endswith("[memory]"):
            by_provenance["memory"].append(text)
        elif message["role"] == "system":
            by_provenance["instructions"].append(text)
        elif message["role"] == "user":
            by_provenance["user"].append(text)

for kind, texts in by_provenance.items():
    print(f"-- {kind} ({len(texts)}):")
    for text in texts:
        print("  ", text[:100])

assert any("42 percent" in text for text in by_provenance["memory"])
assert "holdings.csv" in answer.text

## The committed spine

`RunResult.trace` lists the committed record kinds in journal order —
the durable evidence behind the events and the context above. Every
contribution the model saw corresponds to committed records a later
audit can replay.

In [ ]:
result_trace = answer.trace
print(result_trace)
assert result_trace, "the run committed records"

import shutil

shutil.rmtree(workdir, ignore_errors=True)
print("cleaned", workdir)